# 02b · N3: N4 adaylarının ONNX'e dışa aktarımı

**Amaç:** N2 tur 1'in (`nas_r1b`) N4 aday listesindeki her (mimari, çözünürlük) için cihazda gecikme ölçümüne hazır ONNX dosyası üretmek. N3 = adayların ONNX'e dışa aktarımı, N4 = cihazda gecikme ölçümü (seçim RPi 5 + ORT CPU; RPi gelene kadar Jetson Orin Nano CPU'sunda ön ölçüm, Furkan 2026-10-07/09). Eğitim yok; **CPU runtime yeterli** (GPU kredisi harcamaz).

**Aday listesi** (`results/nas_r1b/n4_candidates.csv`, `python scripts/nas_export.py candidates results/nas_r1b` ile üretilir; 19 aday, 7 mimari):

| Katman | Kural | Aday | Seçime girer mi |
|---|---|---|---|
| `budget` | aynı çözünürlükteki kontrole göre Δ ≥ −1,0 (Karar 5) | 13 | evet |
| `noise_margin` | −1,3 ≤ Δ < −1,0 (ölçülen kontrol gürültüsü 0,3) | 3 | yalnız 2. turdan sonra |
| `reference` | `relu_dfull_k5` × 3 çözünürlük | 3 | hayır — ReLU'nun ORT CPU'da hız kazancı 2. turdaki uzun ince ayara değer mi? |

**Dışa aktarma protokolü** (AGENTS.md; workflow 6. aşama): opset 13 · statik `1×3×H×W` (dinamik eksen yok) · giriş `images`, çıkış `output` (`1×N×85`) · `decode_in_inference=False` (YOLOX `export_onnx.py` varsayılanı; kutu çözme son işlemde) · onnx-simplifier yok (ORT kendi grafik eniyilemesini yapar) · TorchScript dışa aktarıcı (`dynamo=False`). Ağırlık: N2'deki `last_epoch_ckpt.pth` (EMA, epoch 10), mimari cerrahisinden sonra `strict=True` yüklenir. Her aday için PyTorch ↔ ONNX Runtime (4 thread) çıktı eşdeğerliği: 1 rastgele giriş + 4 val2017 görseli (ValTransform ile aynı ön işlem). Ölçüt **kod çözülmüş uzayda** (Furkan, 2026-10-09): kutu merkezi/boyutu farkı ≤ 0,1 piksel (referans skoru ≥ 0,001 olan anchor'larda), skor (obj × cls) farkı ≤ 1e-4; ham maks. fark da kaydedilir. Gerekçe: ham kutu regresyonunda float32 toplama sırası farkı 1e-3'ü aşabiliyor (`relu_dfull_k5_s512`: 3,9e-3 ham = 0,03 px).

**Kayıtlar:** Drive `edge_ai/results/nas_r1b/n3_onnx/` → `manifest.json` (protokol, sürümler, aday listesi), `candidates/<aday>.json` (ckpt/ONNX SHA256, düğüm/op sayıları, eşdeğerlik farkları), `exports.csv`. ONNX dosyaları yalnız Drive'da: `edge_ai/artifacts/onnx/nas_r1b/` + Jetson'a taşımak için `nas_r1b_n4_onnx.zip` (içinde `SHA256SUMS`).

**Kesintide:** hücre 1–7'yi yeniden çalıştır (hücre 4'te Drive bağlantısı elle). Bitmiş adaylar SHA256 eşleşirse atlanır.

Kod: `scripts/nas_export.py` (aday listesi, dışa aktarma, kontroller), `scripts/nas_search.py`, `exps/nas/nano_macro.py`. YOLOX fork'una dokunulmaz.

In [ ]:
# 1 · Ortam ve sürüm kaydı (CPU runtime yeterli)
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · Bağımlılıklar (YOLOX PYTHONPATH ile; bkz. notebook 02 hücre 3) + ONNX araçları
!pip install -q loguru tabulate thop psutil opencv-python-headless onnx onnxruntime

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "onnx", "onnxruntime", "loguru", "thop"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
import importlib, nas_search as ns, nas_export as nx
importlib.reload(ns); importlib.reload(nx)
EXP_FILE = f"{WS}/exps/nas/nano_macro.py"

In [ ]:
# 4 · Drive yolları
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
ROUND = "nas_r1b"
SRC_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"          # N2 kayıtları + checkpoint'ler
OUT = f"{SRC_RESULTS}/n3_onnx"                          # bu notebook'un kayıtları
ONNX_DIR = f"{DRIVE_ROOT}/artifacts/onnx/{ROUND}"      # ONNX dosyaları (repoya girmez)
VAL_ZIP = f"{DRIVE_ROOT}/data/val2017.zip"
for d in (OUT, ONNX_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{SRC_RESULTS}/candidates.csv"), "N2 kayıtları yok"

In [ ]:
# 5 · Aday listesi: depodaki n4_candidates.csv, Drive'daki N2 sonuçlarından yeniden üretilenle aynı olmalı
cands = nx.read_csv(f"{WS}/results/{ROUND}/n4_candidates.csv")
again = nx.select_candidates(nx.read_csv(f"{SRC_RESULTS}/candidates.csv"))
key = lambda rows: [(r["candidate_id"], r["tier"]) for r in rows]
assert key(cands) == key(again), "depodaki aday listesi Drive sonuçlarıyla uyuşmuyor"

CKPT = {r: f"{SRC_RESULTS}/runs/{r}/last_epoch_ckpt.pth" for r in sorted({c["run_id"] for c in cands})}
missing = [p for p in CKPT.values() if not os.path.exists(p)]
assert not missing, missing
from collections import Counter
print(dict(Counter(c["tier"] for c in cands)), f"· {len(cands)} aday · {len(CKPT)} checkpoint")
for c in cands:
    print(f"  {c['tier']:13s} {c['candidate_id']:26s} mAP {c['map5095']:>5s}  Δ {float(c['delta_vs_control']):+.1f}")

In [ ]:
# 6 · Eşdeğerlik görselleri: val2017'nin ad sırasıyla ilk 4 görseli (zip'in tamamı kopyalanmaz)
import zipfile
EQ_DIR = "/content/eq_images"
with zipfile.ZipFile(VAL_ZIP) as z:
    names = sorted(n for n in z.namelist() if n.endswith(".jpg"))[:4]
    for n in names:
        if not os.path.exists(f"{EQ_DIR}/{n}"):
            z.extract(n, EQ_DIR)
EQ_IMAGES = [f"{EQ_DIR}/{n}" for n in names]
print([os.path.basename(p) for p in EQ_IMAGES])

In [ ]:
# 7 · Manifest: dışa aktarma turunun kimlik kartı (aday listesi, protokol, sürümler)
manifest = ns.ensure_manifest(f"{OUT}/manifest.json", {
    "round": ROUND,
    "notebook": "notebooks/02b_aday_onnx_disa_aktarma.ipynb",
    "decisions": "Karar 5 (seçim RPi 5 + ORT CPU); N4 için Jetson CPU ön ölçümü (Furkan, 2026-10-07/09); "
                 "AGENTS.md: opset 13, statik giriş, ORT 4 thread",
    "candidates": [{"candidate_id": c["candidate_id"], "tier": c["tier"]} for c in cands],
    "selection_rule": {"budget": nx.BUDGET, "noise_margin": nx.NOISE_MARGIN,
                       "reference_archs": list(nx.REFERENCE_ARCHS), "tiers": nx.TIERS},
    "export_protocol": {"opset": nx.OPSET, "input": nx.INPUT_NAME, "output": nx.OUTPUT_NAME,
                        "shape": "1x3xRxR static", "decode_in_inference": False, "onnxsim": False,
                        "exporter": "torch.onnx.export (TorchScript, dynamo=False)",
                        "checkpoint": "last_epoch_ckpt.pth (EMA, epoch 10), strict=True",
                        "equivalence": {"inputs": ["random_u0_255"] + [os.path.basename(p) for p in EQ_IMAGES],
                                        "space": "decoded", "max_box_px": nx.EQ_MAX_BOX_PX,
                                        "max_score_diff": nx.EQ_MAX_SCORE, "box_anchor_min_score": nx.EQ_SCORE_CONF,
                                        "ort_threads": nx.ORT_THREADS}},
    "exp_file_sha256": ns.sha256(EXP_FILE),
    "yolox_commit": env["yolox_commit"],
    "versions": {k: env[f"pkg_{k}"] for k in ("torch", "onnx", "onnxruntime", "numpy")},
    "repo_commit_at_creation": env["repo_commit"],
    "env_at_creation": env,
}, keys=nx.EXPORT_FINGERPRINT_KEYS)
print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "export_protocol", "versions")}, indent=1))

In [ ]:
# 8 · Dışa aktarma döngüsü (kaldığı yerden devam eder; ≈ birkaç dk, CPU)
import time
t0 = time.time()
for i, c in enumerate(cands, 1):
    rec = nx.export_candidate(c, EXP_FILE, CKPT[c["run_id"]], ONNX_DIR, OUT, EQ_IMAGES)
    assert rec["ckpt_epoch"] == 10, rec["ckpt_epoch"]
    print(f"✔ {i}/{len(cands)} {rec['candidate_id']:26s} {rec['onnx_bytes'] / 1e6:.2f} MB · "
          f"{rec['n_nodes']} düğüm · ham {max(e['max_abs_diff'] for e in rec['equivalence']):.1e} · "
          f"kutu {max(e['max_box_px'] for e in rec['equivalence']):.3f} px · "
          f"skor {max(e['max_score_diff'] for e in rec['equivalence']):.1e}",
          flush=True)
rows = nx.rebuild_exports_csv(OUT, [c["candidate_id"] for c in cands])
print(f"{len(rows)} kayıt · {time.time() - t0:.0f} sn")

In [ ]:
# 9 · Özet + Jetson'a taşıma paketi: ONNX'ler + aday listesi + kayıtlar + SHA256SUMS
# Jetson'da: unzip nas_r1b_n4_onnx.zip && cd nas_r1b_n4_onnx && sha256sum -c SHA256SUMS
import pandas as pd
df = pd.DataFrame(rows)
print(df[["candidate_id", "tier", "onnx_bytes", "n_nodes", "max_abs_diff", "max_box_px", "max_score_diff"]].to_string(index=False))
print("\nAktivasyon op'ları (opset 13'te HardSwish op'u yok → HardSigmoid + Mul):")
for c in cands:
    if c["resolution"] == "416" or c["tier"] == "reference":
        ops = json.loads(df.set_index("candidate_id").loc[c["candidate_id"], "op_counts"])
        print(f"  {c['candidate_id']:26s}", {k: ops.get(k, 0) for k in ("Conv", "Sigmoid", "Mul", "HardSigmoid", "Relu")})

PKG = f"nas_r1b_n4_onnx"
zip_path = f"{DRIVE_ROOT}/artifacts/onnx/{PKG}.zip"
sums = "".join(f"{r['onnx_sha256']}  {r['onnx_file']}\n" for r in rows)
with zipfile.ZipFile(zip_path + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
    for r in rows:
        z.write(f"{ONNX_DIR}/{r['onnx_file']}", f"{PKG}/{r['onnx_file']}")
    z.write(f"{WS}/results/{ROUND}/n4_candidates.csv", f"{PKG}/n4_candidates.csv")
    z.write(f"{OUT}/exports.csv", f"{PKG}/exports.csv")
    z.write(f"{OUT}/manifest.json", f"{PKG}/manifest.json")
    z.writestr(f"{PKG}/SHA256SUMS", sums)
os.replace(zip_path + ".tmp", zip_path)
print(f"\n{zip_path} · {os.path.getsize(zip_path) / 1e6:.1f} MB · sha256 {ns.sha256(zip_path)}")